# 03. Аналитика для диагностики

Расчёты по обработанным слоям (`data/processed/`) и границам версий (ноутбук 01):

1. Баланс территории v2 (жилая / нежилая / промышленная / зелень / прочее);
2. Застройка по периодам (HOITH) — возрастная структура;
3. Пешая доступность базовых сервисов (школы, детсады, здоровье) — буферный анализ по прямой с оговоркой (сетевой расчёт — этап IduEdu);
4. «Отрыв берега»: доля береговой полосы Невы и Охты, занятая промышленностью/огороженными территориями — ключевой конфликт района.

Все числа диагностического документа (`docs/diagnostics.md`) проверяются `scripts/check_numbers.py`.

In [4]:
# Настройка и загрузка слоёв
from pathlib import Path

import geopandas as gpd
import pandas as pd

PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
RAW, PROC = PROJECT / "data/raw", PROJECT / "data/processed"
CRS_METRIC = 32636

versions = gpd.read_file(PROC / "boundary_versions.geojson").set_index("version")
V2 = versions.loc["v2", "geometry"]
V3 = versions.loc["v3", "geometry"]
AREA_V2 = versions.loc["v2", "area_km2"]
AREA_V3 = versions.loc["v3", "area_km2"]
print(f"v2 {AREA_V2:.2f} км2 | v3 {AREA_V3:.2f} км2")

building = gpd.read_file(PROC / "building.geojson")
green = gpd.read_file(PROC / "green.geojson")
services = gpd.read_file(PROC / "services.geojson")
roads = gpd.read_file(PROC / "roads.geojson")
water = gpd.read_file(PROC / "water.geojson")
house_age = gpd.read_file(PROC / "house_age.geojson")
okn = gpd.read_file(PROC / "okn.geojson")
print("слои загружены")

v2 5.26 км2 | v3 11.06 км2
слои загружены


In [6]:
# Баланс территории v2: что занимает землю района (без воды)
land = V2.difference(water.union_all().intersection(V2).buffer(0))
land_km2 = gpd.GeoSeries([land], crs=4326).to_crs(CRS_METRIC).area.iloc[0] / 1e6

def share_km2(gdf, mask_geom):
    """Суммарная площадь пересечений слоя с территорией, км2 (без наложений внутри слоя)."""
    if not len(gdf):
        return 0.0
    clipped = gpd.GeoSeries(gdf.geometry, crs=4326).intersection(mask_geom)
    clipped = clipped[~clipped.is_empty]
    if not len(clipped):
        return 0.0
    merged = clipped.union_all()
    return gpd.GeoSeries([merged], crs=4326).to_crs(CRS_METRIC).area.iloc[0] / 1e6

# классификация застройки
LIVING = ["apartments", "residential", "dormitory", "detached", "house", "semidetached_house", "yes"]
IND = ["industrial", "warehouse"]
building["kind"] = "прочая"
building.loc[building.building.isin(IND), "kind"] = "промышленно-складская"
building.loc[building.building.isin(LIVING), "kind"] = "жилая"

rows = {}
for kind in ["жилая", "промышленно-складская", "прочая"]:
    rows[kind] = share_km2(building[building.kind == kind], V2)
# зелень: слой green уже отфильтрован по тегам (kind), вычитаем воду
GREEN_EXCL = {"water"}
green_only = green[~green["kind"].isin(GREEN_EXCL)]
rows["зелень"] = share_km2(green_only, V2)
water_v2 = share_km2(water, V2)

print(f"Территория v2 (без воды): {land_km2:.2f} км2, вода: {water_v2:.2f} км2")
for k, v in rows.items():
    print(f"  {k}: {v:.2f} км2 ({100 * v / AREA_V2:.1f} % территории округа)")
print(f"  покрыто всеми слоями: {sum(rows.values()):.2f} км2")

Территория v2 (без воды): 4.46 км2, вода: 0.80 км2
  жилая: 0.36 км2 (6.8 % территории округа)
  промышленно-складская: 0.03 км2 (0.5 % территории округа)
  прочая: 0.23 км2 (4.5 % территории округа)
  зелень: 2.26 км2 (43.0 % территории округа)
  покрыто всеми слоями: 2.88 км2


In [10]:
# «Отрыв берега»: сколько прибрежной полосы занято жилой застройкой.
# Полоса = буфер 300 м от осевых линий Невы и Охты (как в scripts/check_numbers.py — полигональная
# вода в выписке обрезана по bbox, осевые полные), пересечение с v3.
import json
from shapely.geometry import LineString

wp = json.load(open(RAW / "cache/overpass_water_bbox.json", encoding="utf-8"))
axes = [LineString([(g["lon"], g["lat"]) for g in el["geometry"]])
        for el in wp["elements"] if el.get("tags", {}).get("name") in ("Нева", "Охта")]
band = gpd.GeoSeries(axes, crs=4326).to_crs(CRS_METRIC).buffer(300).union_all()
shore_w = gpd.GeoSeries([band], crs=CRS_METRIC).to_crs(4326).iloc[0].intersection(V3)
shore_km = gpd.GeoSeries([shore_w], crs=4326).to_crs(CRS_METRIC).area.iloc[0] / 1e6

# занятость берега: жилая застройка
living_shore = share_km2(living3, shore_w)
print(f"прибрежная полоса 300 м в v3: {shore_km:.2f} км2")
print(f"  жилая застройка в полосе: {living_shore:.2f} км2 ({100 * living_shore / shore_km:.1f} %)")
print(f"  (промзоны по OSM в полосе: {share_km2(ind_v2.clip(V3), shore_w):.2f} км2 — нижняя оценка, слой неполный)")

прибрежная полоса 300 м в v3: 3.75 км2
  жилая застройка в полосе: 0.18 км2 (4.9 %)
  (промзоны по OSM в полосе: 0.13 км2 — нижняя оценка, слой неполный)


In [8]:
# Пешая доступность базовых сервисов (буфер по прямой; сетевой расчёт — этап IduEdu)
from shapely.ops import nearest_points

svc2 = services[services.within(V3)].copy()
schools = svc2[svc2.amenity == "school"]
kinder = svc2[svc2.amenity == "kindergarten"]
health = svc2[svc2.amenity.isin(["clinic", "doctors", "hospital", "pharmacy"])]
print(f"в v3: школ {len(schools)}, детсадов {len(kinder)}, объектов здоровья {len(health)}")

# жилая застройка v3 как получатели услуг
living = building[building.kind == "жилая"]
living3 = living[living.within(V3)]
centroids = living3.to_crs(CRS_METRIC).geometry.centroid.to_crs(4326)

def coverage(poi_gdf, radius_m):
    """Доля жилых зданий в радиусе radius_m от хотя бы одного объекта сервиса (по прямой)."""
    if not len(poi_gdf):
        return 0.0
    zones = poi_gdf.to_crs(CRS_METRIC).buffer(radius_m).union_all()
    zones_w = gpd.GeoSeries([zones], crs=CRS_METRIC).to_crs(4326).iloc[0]
    return 100 * centroids.within(zones_w).mean()

for name, gdf in [("школы (500 м)", schools), ("детсады (300 м)", kinder),
                  ("здоровье (500 м)", health[health.amenity != "pharmacy"]),
                  ("аптеки (300 м)", health[health.amenity == "pharmacy"])]:
    print(f"  {name}: {coverage(gdf, int(name.split('(')[1].split(' ')[0])):.0f} % жилых зданий в зоне")

в v3: школ 23, детсадов 34, объектов здоровья 95
  школы (500 м): 77 % жилых зданий в зоне
  детсады (300 м): 64 % жилых зданий в зоне
  здоровье (500 м): 70 % жилых зданий в зоне
  аптеки (300 м): 56 % жилых зданий в зоне


In [7]:
# Застройка v2 по периодам (HOITH) — возрастная структура жилого фонда
ha2 = house_age[house_age.within(V2)].copy()
known = ha2[ha2.year.notna()]
print(f"домов в v2: {len(ha2)}, с известным годом: {len(known)}")
bins = pd.cut(known.year, [1700, 1917, 1945, 1960, 1980, 1991, 2005, 2027],
              labels=["до 1917", "1917–1945", "1946–1960", "1960–1979", "1981–1991", "1992–2005", "2006+"],
              right=False)
print(bins.value_counts().sort_index().to_dict())
share_pre_rev = 100 * (known.year < 1917).mean()
share_soviet = 100 * known.year.between(1946, 1991).mean()
print(f"до 1917: {share_pre_rev:.1f} % | 1946–1991: {share_soviet:.1f} % домов с известным годом")

# промзоны: землепользование industrial вдоль Охты
landuse_g = green
ind_v2 = landuse_g[(landuse_g.kind == "industrial") & landuse_g.intersects(V2)]
ind_area = share_km2(ind_v2, V2)
print(f"\nпромзоны (landuse=industrial) в v2: {len(ind_v2)} контуров, {ind_area:.2f} км2 ({100 * ind_area / AREA_V2:.1f} %)" )

домов в v2: 635, с известным годом: 478
{'до 1917': 23, '1917–1945': 55, '1946–1960': 73, '1961–1980': 240, '1981–1991': 26, '1992–2005': 18, '2006+': 43}
до 1917: 4.8 % | 1946–1991: 71.1 % домов с известным годом

промзоны (landuse=industrial) в v2: 12 контуров, 0.21 км2 (4.1 %)
